# Scientific Claim Checker — Retrieve, Read and Judge

**P08 group project (Modules 1–4).** Given a scientific claim, the system retrieves relevant abstracts
from the 5,183-paper SciFact corpus, selects the evidence sentences and answers **SUPPORT**,
**CONTRADICT** or **NEI** (not enough information) with the evidence quoted exactly.

This notebook is the single assessment notebook. It **imports the shared project package**
(`src/claim_checker`), the same code used by the command line and the Streamlit app, so the assessment uses the shared implementation. The explicitly marked
teaching appendix in §12 adapts actual course-lab examples separately.

**How this notebook runs.** Data acquisition runs at the top with `load_dataset(...)`, as the brief
requires. Every expensive experiment (index build, six-way retrieval comparison, 8 variants × 100
claims) was run with the project CLI and saved under `artifacts/`; this notebook **loads those saved
artifacts** and shows exactly where they came from. Only cheap live steps run here: one retrieval demo
on two claims and one live claim check. From a clean kernel it completes in a few minutes (if the
prepared data or the index are missing, it builds them first, which takes longer).

| § | Section | Lab topic (see §10) |
|---|---|---|
| 0 | Environment | — |
| 1 | Data acquisition (`load_dataset`) and the compatibility fallback | — |
| 2 | Data quality checks | — |
| 3 | Frozen evaluation populations | — |
| 4 | Retrieval: BM25 vs dense vs hybrid, ± re-ranking | Module 3 |
| 5 | Agents and the supervised graph | Modules 2 and 4 |
| 6 | Oracle and full-system experiments (one results table) | Modules 3 and 4 |
| 7 | Live claim check | — |
| 8 | Failure analysis | — |
| 9 | Reproducibility record | — |
| 10 | Lab citations | — |
| 11 | Limitations and future work | — |

Run it with the conda base kernel (`python3`), started from the conda interpreter:
`C:\Users\Bhargav\anaconda3\python.exe -m jupyter lab notebooks/Scientific_Claim_Checker.ipynb`.

Sections 0–11 retain the original Qwen assessment and saved results. Later NLI
experiments, reused evaluation populations and the current score status are
identified in §13; the original 100-claim subset is not a blind test anymore.


## 0. Environment

The cell below locates the project root, makes sure the `claim_checker` package can be imported, and
prints the interpreter and library versions. The verified environment is Python 3.14.6 (Anaconda base)
with datasets 5.1.0, sentence-transformers 6.1.0, torch 2.14.1 (CPU), langgraph 1.2.16 and the
`ollama` client 0.6.3, talking to a local Ollama 0.40.6 server running `qwen3:4b`.

In [ ]:
import json
import os
import platform
import sys
import textwrap
import time
from importlib import metadata
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Locate the project root (the folder holding configs/default.yaml) and work from there.
ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "configs" / "default.yaml").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from inside the project folder (configs/default.yaml not found).")
os.chdir(ROOT)

try:
    import claim_checker
except ImportError:  # not installed: fall back to the source checkout
    sys.path.insert(0, str(ROOT / "src"))
    try:
        import claim_checker
    except ImportError as exc:
        raise RuntimeError(
            f"claim_checker is not importable from {sys.executable}. Install it with\n"
            '  C:\\Users\\Bhargav\\anaconda3\\python.exe -m pip install -e ".[dev]"\n'
            "and start Jupyter with that interpreter."
        ) from exc

print("Interpreter :", sys.executable)
print("Python      :", platform.python_version(), "|", platform.platform())
print("Project root:", ROOT)
print("claim_checker", claim_checker.__version__)
for dist in ["datasets", "huggingface_hub", "torch", "sentence-transformers", "transformers", "rank-bm25",
             "langgraph", "ollama", "pydantic", "scikit-learn", "numpy", "pandas", "streamlit"]:
    try:
        print(f"  {dist:22s} {metadata.version(dist)}")
    except metadata.PackageNotFoundError:
        print(f"  {dist:22s} (not installed)")

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

from claim_checker.config import load_config, load_experiments
from claim_checker.data.store import CORPUS_FILE, load_corpus, load_gold, load_pairs, read_json
from claim_checker.schemas import normalize_doc_id

cfg = load_config()
experiments = load_experiments()

# Settings for this notebook
RUN_ID = "final_v1"           # saved final evaluation run (python -m claim_checker evaluate --suite final --run-id final_v1)
SMOKE_RUN_ID = "smoke_v1"     # saved development smoke run (training claims)
LIVE_DEMOS = True             # run the cheap live retrieval demo and the live claim check
BUILD_INDEX_IF_MISSING = True # build the retrieval index if it does not exist (~8 min on CPU)

print("Model     :", cfg.model.name, "at", cfg.model.host, "| temperature", cfg.model.temperature,
      "| think", cfg.model.think, "| num_ctx", cfg.model.context_tokens)
print("Embedding :", cfg.retrieval.embedding_model, "| re-ranker:", cfg.retrieval.reranker_model)
print("Retrieval : candidates/method", cfg.retrieval.candidates_per_method, "| RRF k", cfg.retrieval.rrf_k,
      "| re-rank top", cfg.retrieval.rerank_top_n, "| keep", cfg.retrieval.keep_top, "| pipeline k", cfg.retrieval.pipeline_k)

## 1. Data acquisition

The brief specifies two Hugging Face datasets:

| Dataset | Config / split | Columns | Role |
|---|---|---|---|
| `mteb/scifact` | `corpus` / `corpus` | `_id`, `title`, `text` | 5,183 abstracts — the knowledge base |
| `allenai/scifact_entailment` | default / train, validation | claim, abstract_id, title, abstract, verdict, evidence | claim–abstract pairs with gold verdicts and evidence sentence positions |

### 1.1 The retrieval corpus

This is the exact call from the brief. The retrieval index used later is built from this download.

In [ ]:
from datasets import load_dataset

corpus = load_dataset("mteb/scifact", "corpus")["corpus"]
print(corpus)
example = corpus[0]
print("\nFirst row:", {k: (v[:160] + "...") if isinstance(v, str) and len(v) > 160 else v for k, v in example.items()})

### 1.2 The claims dataset — attempted exactly as in the brief

`allenai/scifact_entailment` is a *script-based* dataset (it ships a Python builder,
`scifact_entailment.py`). Recent releases of `datasets` no longer execute dataset scripts. The cell
below runs the brief's call and **displays the exception** instead of hiding it. Remote code is never
enabled (`trust_remote_code` is not used).

In [ ]:
try:
    claims = load_dataset("allenai/scifact_entailment")
    ENTAILMENT_LOADER_OK = True
    print(claims)
except Exception as exc:
    ENTAILMENT_LOADER_OK = False
    print("load_dataset('allenai/scifact_entailment') failed with datasets", metadata.version("datasets"))
    print(f"{type(exc).__module__}.{type(exc).__qualname__}: {exc}")

### 1.3 Compatibility fallback (implemented in `claim_checker.data.sources`)

When the loader fails, the package follows a recorded route, `fallback_official_archive_json`:

1. Record the exact exception.
2. Download the builder script **as text only** (never executed) and read the archive URL (`_URL`) it
   uses: `https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz`.
3. Download that official SciFact archive (sha256 recorded) and extract only `corpus.jsonl`,
   `claims_train.jsonl` and `claims_dev.jsonl`.
4. Read them with `load_dataset("json", ...)` and cross-check against a plain JSON parse.
5. Reproduce the builder's claim–abstract transformation in `claim_checker.data.prepare`
   (one row per claim × cited document; verdict from that document's evidence sets, otherwise NEI;
   evidence = flattened zero-based sentence positions).
6. Verify IDs, labels, evidence positions and counts; write `data/raw/source_manifest.json`.

The next cell runs the same steps as `python -m claim_checker prepare-data` **only if** the prepared
files are missing; otherwise it reuses them (preparation is deterministic and the final split is
frozen, so re-running would not change anything).

In [ ]:
from types import SimpleNamespace

processed_dir = cfg.path("processed_dir")
raw_dir = cfg.path("raw_dir")
splits_dir = cfg.path("splits_dir")
needed = [processed_dir / CORPUS_FILE, raw_dir / "source_manifest.json", splits_dir / "final.json",
          processed_dir / "claim_gold_validation.jsonl", processed_dir / "data_quality.json"]
missing = [p for p in needed if not p.exists()]
if missing:
    print("Prepared data missing:", [str(p.relative_to(ROOT)) for p in missing])
    print("Running the prepare-data pipeline (needs internet once)...")
    from claim_checker.cli import cmd_prepare_data

    cmd_prepare_data(cfg, SimpleNamespace(force_download=False, force_splits=False))
else:
    print("Prepared data found - reusing data/processed, data/splits and data/raw/source_manifest.json")

source_manifest = read_json(raw_dir / "source_manifest.json")
m = source_manifest
rows = [
    ("Route", m["route"]),
    ("Brief loader call", m["entailment_loader"]["call"]),
    ("Brief loader succeeded", m["entailment_loader"]["success"]),
    ("Recorded exception", m["entailment_loader"].get("exception", "")),
    ("Builder script revision", m["builder_script"].get("revision")),
    ("Builder script executed?", m["builder_script"].get("executed")),
    ("Archive URL source", m["builder_script"].get("archive_url_source")),
    ("Archive URL", m["archive"]["url"]),
    ("Archive sha256", m["archive"]["sha256"]),
    ("Archive bytes", f"{m['archive']['bytes']:,}"),
    ("MTEB call", m["mteb_corpus"]["call"]),
    ("MTEB revision", m["mteb_corpus"]["revision"]),
    ("MTEB rows", m["mteb_corpus"]["rows"]),
    ("Rows read with load_dataset('json')", m["json_loader_rows"]),
    ("Environment", m["environment"]),
]
display(pd.DataFrame(rows, columns=["field", "value"]))
display(pd.DataFrame(m["extracted"]).T)

The original claim files, read with `load_dataset("json", ...)` exactly as the fallback does. The
`evidence` column becomes a wide Arrow struct (one field per document ID), so it is shown here after
the package's normalisation.

In [ ]:
from claim_checker.data.prepare import normalize_claim_row

scifact_dir = raw_dir / "scifact"
claims_json = {
    "train": load_dataset("json", data_files=str(scifact_dir / "claims_train.jsonl"), split="train"),
    "validation": load_dataset("json", data_files=str(scifact_dir / "claims_dev.jsonl"), split="train"),
}
for name, ds in claims_json.items():
    print(f"{name:10s} {ds.num_rows} claims, columns {ds.column_names}")

row = normalize_claim_row(claims_json["validation"][0])
print("\nFirst validation claim after normalisation:")
print(json.dumps(row, indent=1)[:800])

### 1.4 Reconstructed records and corpus alignment

The package joins the MTEB corpus to the original sentence-segmented corpus by normalised document ID
(`13734012`, `"13734012"` and `" 013734012 "` are the same document). MTEB supplies the retrieval
text; the original archive supplies the sentence list from which every quotation is copied. Sentence
positions are **zero-based** internally; only displays add 1.

In [ ]:
docs = load_corpus(processed_dir)
mteb_ids = {normalize_doc_id(x) for x in corpus["_id"]}
print(f"Processed corpus: {len(docs):,} documents | MTEB download: {len(mteb_ids):,} ids | identical id sets: {mteb_ids == set(docs)}")

pairs_val = load_pairs(processed_dir, "validation")
gold_train = load_gold(processed_dir, "train")
gold_val = load_gold(processed_dir, "validation")
print(f"Claim-abstract pairs: validation {len(pairs_val)} | claims: train {len(gold_train)}, validation {len(gold_val)}")

display(pd.DataFrame([p.model_dump() for p in pairs_val[:6]]))

# One pair, with its evidence sentence recovered from the stored sentence list (zero-based positions)
g = gold_train["34"]
doc_id, sent_ids = next(iter(g.evidence_by_doc.items()))
print(f"\nTraining claim {g.claim_id}: {g.claim!r} -> label {g.label}")
print(f"Reference document {doc_id}: {docs[doc_id].title}")
for i in sent_ids:
    print(f"  sentence_id {i} (displayed as sentence {i + 1}): {docs[doc_id].sentences[i]}")

## 2. Data quality checks

`prepare-data` fails loudly on unexplained problems (duplicate or missing IDs, title mismatches,
evidence positions out of range, evidence sets that disagree) instead of silently dropping rows. The
report it writes is shown below.

In [ ]:
quality = read_json(processed_dir / "data_quality.json")
cq = quality["corpus"]
corpus_checks = {
    "MTEB documents": cq["mteb_documents"],
    "original documents": cq["original_documents"],
    "ids only in MTEB": len(cq["ids_only_in_mteb"]),
    "ids only in original": len(cq["ids_only_in_original"]),
    "title mismatches": len(cq["title_mismatches"]),
    "text equal after whitespace normalisation": cq["text_agreement"]["exact_after_whitespace_normalization"],
    "text equal ignoring all whitespace": cq["text_agreement"]["equal_ignoring_whitespace"],
    "text different": cq["text_agreement"]["different"],
    "empty abstracts": len(cq["empty_abstracts"]),
    **{f"sentences per document ({k})": v for k, v in cq["sentences_per_document"].items()},
}
display(pd.Series(corpus_checks, name="value").to_frame())

split_rows = []
for split, s in quality["splits"].items():
    split_rows.append({
        "split": split, "claims": s["claims"], "claim-abstract pairs": s["pairs"],
        "pair verdicts": s["pair_verdicts"], "claim labels": s["claim_labels"],
        "annotation conflicts": len(s["annotation_conflicts"]),
        "cited docs missing from corpus": s["cited_docs_missing_from_corpus"],
        "evidence positions out of range": s["evidence_positions_out_of_range"],
        "reference evidence sentences": s["evidence_sentences"],
    })
display(pd.DataFrame(split_rows))

In [ ]:
import matplotlib.pyplot as plt

n_sent = pd.Series([len(d.sentences) for d in docs.values()])
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(n_sent.clip(upper=40), bins=38, color="#4C72B0")
ax.set_xlabel("sentences per abstract (values above 40 shown at 40)")
ax.set_ylabel("abstracts")
ax.set_title(f"SciFact corpus: {len(n_sent):,} abstracts, {n_sent.sum():,} sentences "
             f"(min {n_sent.min()}, median {int(n_sent.median())}, max {n_sent.max()})")
plt.tight_layout()
plt.show()

## 3. Frozen evaluation populations

* **final** — 100 unique validation claims, seed 42, stratified by claim-level label (from the 300
  eligible; no annotation conflicts exist). Frozen: `prepare-data` refuses to overwrite it.
* **development** — training claims, minus any claim whose normalised text appears in the final
  subset (train 871 duplicates final 870 and is removed; train 1291 overlaps a validation claim that is
  not in the final subset and is kept).
* **smoke** — 20 development claims for early real-model checks.

Prompts and settings were developed on training claims only.

Claim-level labels: SUPPORT if annotated evidence only supports, CONTRADICT if it only contradicts, NEI
if no document has annotated evidence. Unannotated documents are not proven irrelevant.

In [ ]:
from claim_checker.data.splits import load_split

splits = {name: load_split(cfg, name) for name in ("final", "development", "smoke")}
display(pd.DataFrame([{
    "split": n, "source": s["source_split"], "claims": s["size"], "class counts": s["class_counts"],
    "seed": s.get("seed", ""), "ids sha256": s["ids_sha256"][:16] + "...",
} for n, s in splits.items()]))

final = splits["final"]
print(f"Final: {final['size']} of {final['eligible_claims']} eligible validation claims; "
      f"excluded annotation conflicts: {final['excluded_annotation_conflicts']}; allocation {final['allocation']}")
print("Development overlap removed:", splits["development"]["removed_overlap_with_final"])
print("Overlap with non-final validation claims (kept):", splits["development"]["overlap_with_other_validation_claims_kept"])
print("\nFrozen final claim IDs:")
print(textwrap.fill(" ".join(final["claim_ids"]), 110))

final_gold = [gold_val[c] for c in final["claim_ids"]]
print(f"\nFinal claims with evidence-bearing documents: {sum(1 for g in final_gold if g.evidence_doc_ids)} "
      f"| reference documents per claim: {pd.Series([len(g.reference_doc_ids) for g in final_gold]).value_counts().sort_index().to_dict()}")

## 4. Retrieval

*Builds on Module 3 (RAG_Pipeline_Lab) topics: chunking, BM25, hybrid RRF, cross-encoder re-ranking,
Recall@k — see §10.*

* **BM25** over `title + abstract` with a deterministic scientific tokenizer (keeps numbers and
  hyphenated terms such as `il-6`, plus their parts), ties broken by document ID.
* **Dense**: `BAAI/bge-small-en-v1.5`, query prefixed with the BGE instruction. Long abstracts are
  split into tokenizer-aware sentence windows (≤ 512 tokens, one sentence overlap) that remember their
  zero-based sentence IDs; a document's score is its best window. Exact inner product over a NumPy
  matrix (no FAISS needed at this size).
* **Hybrid**: reciprocal rank fusion of the BM25 and dense top-30 lists, `1/(60 + rank)`.
* **Re-ranking**: `cross-encoder/ms-marco-MiniLM-L6-v2` re-scores the first 20 candidates (long
  abstracts windowed, maximum window score). Scores mean relevance, not truth. The top 10 are kept for
  metrics; the top `k = 3` are read by the agents.

The index was built from the corpus downloaded in §1 (`python -m claim_checker build-index`). Its
manifest records a fingerprint of everything that would make it stale.

In [ ]:
from claim_checker.errors import ClaimCheckerError
from claim_checker.retrieval.index import check_index

try:
    index_manifest = check_index(cfg)
    print("Index is present and fresh.")
except ClaimCheckerError as exc:
    print(f"Index not ready: {exc.message}")
    if not BUILD_INDEX_IF_MISSING:
        raise
    from claim_checker.retrieval.index import build_index

    print("Building the index from the prepared corpus (about 8 minutes on CPU)...")
    index_manifest = build_index(cfg, docs)

display(pd.Series({
    "fingerprint": index_manifest["fingerprint"],
    "documents": index_manifest["documents"],
    "windows": index_manifest["windows"],
    "documents with >1 window": index_manifest["documents_with_multiple_windows"],
    "truncated single-sentence windows": index_manifest["truncated_single_sentence_windows"],
    "embedding dim": index_manifest["embedding_dim"],
    "embedding model revision": index_manifest.get("embedding_model_revision"),
    "build seconds": index_manifest.get("timings_seconds"),
    "identity": index_manifest["identity"],
}, name="value").to_frame())

### 4.1 Six-way retrieval comparison (saved results)

Produced by `python -m claim_checker retrieval-eval --split final` on the 100 frozen final claims.

* **Recall@k** — mean over claims of the share of reference (cited) abstracts in the top k
  (100 claims).
* **Evidence-bearing Recall@k** — the same, counting only abstracts with annotated evidence, over the
  62 claims that have such abstracts (the 41 SUPPORT + 21 CONTRADICT claims).
* **Latency** — seconds per claim on CPU (search, plus re-ranking when on).

In [ ]:
retrieval_path = cfg.path("retrieval_dir") / "retrieval_eval_final.json"
if retrieval_path.exists():
    retrieval_eval = read_json(retrieval_path)
    print(f"Saved {retrieval_eval['created_at']} | split {retrieval_eval['split']} | {retrieval_eval['n_claims']} claims "
          f"| index {retrieval_eval['index_fingerprint']}")
    ret_rows = []
    for method, mm in retrieval_eval["results"].items():
        ret_rows.append({
            "method": method,
            **{f"R@{k}": mm[f"recall@{k}"] for k in (1, 3, 5, 10)},
            **{f"EvR@{k}": mm[f"evidence_doc_recall@{k}"] for k in (1, 3, 5, 10)},
            "claims (R)": mm["recall@3_n_claims"], "claims (EvR)": mm["evidence_doc_recall@3_n_claims"],
            "mean s": mm["latency"]["mean"], "p95 s": mm["latency"]["p95"],
        })
    ret_df = pd.DataFrame(ret_rows).set_index("method")
    display(ret_df.style.format("{:.3f}", subset=[c for c in ret_df.columns if c.startswith(("R@", "EvR@", "mean", "p95"))])
            .highlight_max(subset=[c for c in ret_df.columns if c.startswith(("R@", "EvR@"))], color="#d4edda"))

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
    ret_df[["R@3", "R@10"]].plot.bar(ax=axes[0], rot=20, title="Reference-document recall (100 claims)")
    ret_df[["EvR@3", "EvR@10"]].plot.bar(ax=axes[1], rot=20, title="Evidence-bearing recall (62 claims)")
    for ax in axes:
        ax.set_ylim(0.5, 1.0)
        ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    ret_df = None
    print("No saved retrieval comparison found. Create it with:\n"
          "  C:\\Users\\Bhargav\\anaconda3\\python.exe -m claim_checker retrieval-eval --split final")

**Reading the comparison.** Dense retrieval has the best Recall@3 (0.718); hybrid RRF has the best
Recall@10 (0.868) and evidence-bearing Recall@10 (0.964). Re-ranking does **not** improve Recall@3 for
any first stage, but it improves Recall@1 for all three and moves evidence-bearing abstracts up for
BM25 and hybrid (hybrid evidence-bearing Recall@3 0.874 → 0.910), at about 1 s per claim on CPU. The
product keeps hybrid + re-rank because the agents only read the top 3 and evidence-bearing abstracts
matter most for the verdict; the `rerank_off` variant (§6) measures the end-to-end effect. The
re-ranker was trained on MS MARCO web search rather than biomedical claims, which is a plausible reason
for the weak Recall@3 effect.

### 4.2 Live retrieval demo (two training claims)

A small live check that the saved numbers come from working code. It loads the corpus, BM25, the BGE
encoder and the cross-encoder (from the local cache) and prints the top 5 of each method. `*` marks a
reference (cited) abstract; `**` marks one with annotated evidence. These are training claims, so the
final split is not touched.

In [ ]:
from claim_checker.runtime import ClaimChecker

checker = ClaimChecker(cfg, experiments=experiments)  # nothing heavy is loaded until first use


def mark(doc_id, g):
    return f"{doc_id}{'**' if doc_id in g.evidence_doc_ids else '*' if doc_id in g.reference_doc_ids else ''}"


if LIVE_DEMOS:
    t0 = time.perf_counter()
    retriever = checker.retriever
    print(f"Loaded retriever (index {retriever.fingerprint}) in {time.perf_counter() - t0:.1f}s")
    for cid in ("34", "2"):
        g = gold_train[cid]
        cand = retriever.candidates(g.claim)
        reranked = retriever.retrieve(g.claim, method="hybrid", rerank=True, keep=5)
        table = {m_: [mark(d, g) for d, _ in cand.candidates[m_][:5]] for m_ in ("bm25", "dense", "hybrid")}
        table["hybrid+rerank"] = [mark(h.doc_id, g) for h in reranked.hits]
        print(f"\nTraining claim {cid} ({g.label}): {g.claim}")
        print(f"Reference documents: {g.reference_doc_ids} | with evidence: {g.evidence_doc_ids}")
        display(pd.DataFrame(table, index=[f"rank {i}" for i in range(1, 6)]))
        print("timings (s):", {k: round(v, 3) for k, v in {**cand.timings, **reranked.timings}.items()})
else:
    print("LIVE_DEMOS is False - skipped.")

## 5. Agents and the supervised graph

*Builds on Module 2 (structured generation, reflection) and Module 4 (multi-agent graph, supervisor)
topics — see §10.*

```text
validate_input -> retrieve -> select_evidence -> judge -> aggregate -> supervise -> explain -> END
                                                   ^                      |
                                                   +-- repair once -------+
any node failing operationally -> handle_error -> END   (status=error, verdict=None)
```

* **Evidence agent** (`prompts/evidence.jinja2`): sees one abstract with numbered sentences and
  returns `{"sentence_ids": [...], "note": ...}`. It returns IDs only; **the code copies the
  quotation from the stored sentence**, so quotations cannot be invented. IDs outside the supplied list
  trigger one repair attempt and are then dropped. Abstracts with more than 40 sentences are processed
  in bounded windows.
* **Judge** (`prompts/judge.jinja2`): per abstract, sees only the selected sentences (or the whole
  abstract in the `evidence_off` ablation) and returns verdict, cited sentence IDs, a short reason and
  a self-reported (uncalibrated) confidence. Cited IDs must be among the sentences it was shown. If no
  sentence was selected, the abstract is NEI without a model call.
* **Structured output**: every model call sends a JSON schema to Ollama (`format=`), validates the
  answer with Pydantic, allows one repair turn, and raises a typed error otherwise. `think=false`,
  temperature 0, seed 42.
* **Aggregate** (deterministic): SUPPORT only → SUPPORT; CONTRADICT only → CONTRADICT; all NEI → NEI;
  both → NEI with `conflict=true`.
* **Supervisor**: checks every citation; a decisive verdict without a valid citation is sent back to
  the judge once with feedback, then downgraded to NEI. So every successful SUPPORT/CONTRADICT has at
  least one valid quotation. Operational failures (Ollama down, stale index, invalid output, context
  overflow) end with `status="error"` and no verdict — never NEI.
* **Explainer**: deterministic plain-English text from the validated decisions (no model call).

In [ ]:
from claim_checker.pipeline.aggregate import aggregate_verdicts

display(pd.DataFrame([
    {"document verdicts": " + ".join(v), "final": aggregate_verdicts(v)[0], "conflict": aggregate_verdicts(v)[1]}
    for v in (["SUPPORT", "NEI", "NEI"], ["CONTRADICT", "NEI"], ["NEI", "NEI", "NEI"], ["SUPPORT", "CONTRADICT", "NEI"])
]))

print("Prompt template hashes (recorded in every result and run manifest):", checker.prompts.hashes())
print("\n--- evidence.jinja2 (system part) ---")
print(checker.prompts.source("evidence").split("---USER---")[0].strip())
print("\n--- judge.jinja2 (system part) ---")
print(checker.prompts.source("judge").split("---USER---")[0].strip())

try:
    print("\n--- LangGraph structure (Mermaid) ---")
    print(checker._graph.get_graph().draw_mermaid())
except Exception as exc:  # drawing is optional
    print("Graph drawing unavailable:", exc)

## 6. Oracle and full-system experiments

*Builds on Module 3 (retrieval evaluation) and Module 4 (end-to-end multi-agent evaluation) topics.*

All variants ran on the **same 100 frozen final claims** with
`python -m claim_checker evaluate --suite final --run-id final_v1` (resumable; each run records the
data, split, index, model digest, prompt and configuration fingerprints and refuses to resume if any
changed).

| Variant | What it tests |
|---|---|
| `judge_only` | Baseline: the claim alone, no retrieval |
| `retrieve_judge_single_prompt` | Baseline: top-3 re-ranked abstracts judged and cited in one prompt |
| `oracle` | Reference abstracts supplied (IDs only — never gold verdicts or evidence): isolates judgement quality |
| `full` | Main system: hybrid + re-rank, k = 3, evidence agent on |
| `evidence_off` | Ablation: judge reads whole abstracts |
| `rerank_off` | Ablation: hybrid order without the cross-encoder |
| `full_k1`, `full_k5` | Ablations: 1 or 5 abstracts |

Metric rules: accuracy and macro-F1 are over **all requested claims** (operational errors count as
wrong; coverage is shown separately); macro-F1 averages the three classes; evidence precision counts
only sentences in annotated abstracts (unannotated ones are not proven irrelevant) and recall is
end-to-end; latency is seconds per claim and is affected by the LLM response cache (later variants
reuse identical model calls, see `llm_calls` vs `llm_cache_hits`).

The cell below **reads the saved predictions** and recomputes the metrics in memory with the package's
own metric functions (the same ones that wrote `metrics.json` / `results.csv`). If the run is still in
progress it says so and labels the numbers as partial.

In [ ]:
from claim_checker.evaluation.analysis import (
    downgrades_from_events,
    load_predictions,
    results_rows,
    select_failures,
    variant_metrics,
)

RUN_DIR = cfg.path("runs_dir") / RUN_ID
run_metrics, run_rows, by_variant, run_manifest, RUN_COMPLETE = None, None, None, None, False

if (RUN_DIR / "predictions.jsonl").exists():
    run_manifest = read_json(RUN_DIR / "manifest.json")
    by_variant = load_predictions(RUN_DIR)
    expected_variants = list(run_manifest.get("variants", {}))
    size = run_manifest["identity"]["population_size"]
    done = {v: len(by_variant.get(v, [])) for v in expected_variants}
    RUN_COMPLETE = all(n == size for n in done.values())
    print(f"Run {RUN_ID}: created {run_manifest['created_at']} | {size} claims per variant | "
          f"model {run_manifest['identity']['model']['name']} ({run_manifest['identity']['model']['digest'][:12]}...) | "
          f"index {run_manifest['identity']['index_fingerprint']}")
    print("Completed claims per variant:", done)
    if not RUN_COMPLETE:
        print("\n*** PARTIAL RUN - the evaluation has not finished; the numbers below are provisional. ***")
    # Downgrades are also recovered from the saved stage events: final_v1's single-prompt records were
    # written before the claim-level downgrade flag reached predictions.jsonl.
    run_downgrades = downgrades_from_events(RUN_DIR)
    run_metrics = {v: variant_metrics(recs, docs, run_manifest["variants"].get(v, {}).get("strategy"), run_downgrades)
                   for v, recs in by_variant.items()}
    run_rows = results_rows(run_metrics, run_manifest.get("variants", {}))
    cols = ["variant", "n_claims", "coverage", "accuracy", "macro_f1", "f1_support", "f1_contradict", "f1_nei",
            "recall@3", "recall@10", "evidence_precision_selected", "evidence_recall_selected",
            "evidence_precision_cited", "evidence_recall_cited", "quote_validity", "conflict_rate",
            "failure_rate", "latency_mean_s", "latency_p95_s"]
    results_df = pd.DataFrame(run_rows)[cols].set_index("variant")
    display(Markdown(f"**Main results table — run `{RUN_ID}`{'' if RUN_COMPLETE else ' (PARTIAL)'}**"))
    display(results_df.style.format(lambda x: "-" if x is None or x != x else (f"{x:.3f}" if isinstance(x, float) else x)))
    if (RUN_DIR / "results.csv").exists():
        print(f"Saved table: {(RUN_DIR / 'results.csv').relative_to(ROOT)} (report: {(RUN_DIR / 'report.md').relative_to(ROOT)})")
else:
    print(f"No saved run '{RUN_ID}' yet. Run:\n"
          f"  C:\\Users\\Bhargav\\anaconda3\\python.exe -m claim_checker evaluate --suite final --run-id {RUN_ID}\n"
          "then re-run this section.")

In [ ]:
if run_metrics:
    pc_rows = []
    for v, mm in run_metrics.items():
        for label, pc in mm["verdict"].get("per_class", {}).items():
            pc_rows.append({"variant": v, "class": label, "precision": pc["precision"], "recall": pc["recall"],
                            "F1": pc["f1"], "gold claims": pc["support"]})
    pc_df = pd.DataFrame(pc_rows)
    display(Markdown("**Per-class verdict performance (unit: claims)**"))
    display(pc_df.pivot(index="variant", columns="class", values="F1").round(3))

    show = [v for v in ("judge_only", "retrieve_judge_single_prompt", "oracle", "full") if v in run_metrics]
    fig, axes = plt.subplots(1, len(show), figsize=(4.2 * len(show), 3.6))
    axes = [axes] if len(show) == 1 else axes
    for ax, v in zip(axes, show):
        cm = run_metrics[v]["verdict"]["confusion"]["matrix"]
        golds, preds = list(cm), ["SUPPORT", "CONTRADICT", "NEI", "ERROR"]
        mat = [[cm[gl][p] for p in preds] for gl in golds]
        ax.imshow(mat, cmap="Blues")
        ax.set_xticks(range(4), ["S", "C", "N", "ERR"])
        ax.set_yticks(range(3), ["S", "C", "N"])
        ax.set_xlabel("predicted")
        ax.set_ylabel("gold")
        ax.set_title(v, fontsize=10)
        for i, r in enumerate(mat):
            for j, val in enumerate(r):
                ax.text(j, i, val, ha="center", va="center", color="black")
    plt.suptitle(f"Confusion matrices, run {RUN_ID}{'' if RUN_COMPLETE else ' (partial)'} - unit: claims")
    plt.tight_layout()
    plt.show()

In [ ]:
if run_metrics:
    ev_rows, op_rows = [], []
    for v, mm in run_metrics.items():
        for key in ("evidence_selected", "evidence_cited"):
            e = mm.get(key)
            if e:
                ev_rows.append({"variant": v, "sentences": key.split("_")[1], "precision": e["precision"],
                                "TP": e["true_positive_sentences"],
                                "predicted in annotated docs": e["predicted_sentences_in_annotated_docs"],
                                "recall": e["recall"], "reference sentences": e["reference_sentences"],
                                "predicted in unannotated docs (excluded)": e["predicted_sentences_in_unannotated_docs"]})
        op = mm["operational"]
        recs = by_variant[v]
        op_rows.append({"variant": v, "requested": op["requested"], "ok": op["ok"], "errors": op["errors"],
                        "errors by type": op["errors_by_type"], "conflicts": op["conflicts"],
                        "downgraded claims": mm["downgraded_claims"],
                        "LLM calls": sum(r.get("llm_calls") or 0 for r in recs),
                        "LLM cache hits": sum(r.get("llm_cache_hits") or 0 for r in recs),
                        "mean s/claim": mm["latency_total"]["mean"]})
    display(Markdown("**Evidence sentences (micro; unit: evidence sentences)**"))
    display(pd.DataFrame(ev_rows).round(3))
    display(Markdown("**Operational summary (unit: claims). Latency is low where answers came from the cache.**"))
    display(pd.DataFrame(op_rows).round(3))

### 6.1 Development smoke run (training claims — not a final result)

`smoke_v1` ran `full` and `oracle` on the 20 smoke claims from the training split to check the
pipeline end to end before the final run. It is shown for transparency only; 20 claims are too few for
conclusions.

In [ ]:
smoke_csv = cfg.path("runs_dir") / SMOKE_RUN_ID / "results.csv"
if smoke_csv.exists():
    smoke_df = pd.read_csv(smoke_csv)
    display(smoke_df[["variant", "n_claims", "coverage", "accuracy", "macro_f1", "f1_support", "f1_contradict",
                      "f1_nei", "quote_validity", "conflict_rate", "failure_rate"]].round(3))
else:
    print(f"No saved smoke run '{SMOKE_RUN_ID}'.")

### 6.2 Interpretation

Numbers from the saved run `final_v1` (100 claims per variant; the table above recomputes them from
`predictions.jsonl`). Approximate 95 % intervals: percentile bootstrap over claims (seed 42, 1000
resamples, same resamples for every variant; they ignore model non-determinism).

| Variant | Accuracy [95 % CI] | Macro-F1 [95 % CI] | F1 S / C / N |
|---|---|---|---|
| judge_only | 0.500 [0.410, 0.590] | 0.504 [0.395, 0.599] | 0.545 / 0.529 / 0.436 |
| retrieve_judge_single_prompt | 0.560 [0.470, 0.660] | 0.450 [0.350, 0.556] | 0.650 / 0.174 / 0.526 |
| oracle | 0.670 [0.570, 0.760] | 0.571 [0.477, 0.670] | 0.706 / 0.240 / 0.767 |
| **full** | 0.560 [0.460, 0.660] | 0.487 [0.373, 0.597] | 0.638 / 0.296 / 0.526 |
| evidence_off | 0.530 [0.430, 0.630] | 0.446 [0.332, 0.553] | 0.639 / 0.320 / 0.377 |
| rerank_off | 0.540 [0.440, 0.640] | 0.468 [0.357, 0.575] | 0.626 / 0.296 / 0.483 |
| full_k1 | 0.560 [0.470, 0.660] | 0.473 [0.373, 0.579] | 0.660 / 0.214 / 0.545 |
| full_k5 | 0.570 [0.470, 0.670] | 0.493 [0.380, 0.603] | 0.655 / 0.296 / 0.526 |

* **Oracle vs full (cost of retrieval): +0.11 accuracy, +0.08 macro-F1** — the only difference clearly
  beyond noise (paired CI for accuracy [+0.01, +0.19]; 17 vs 6 discordant claims, exact McNemar
  p = 0.035). 16 of the 17 claims that only the oracle gets right are gold **NEI**: retrieval rarely
  misses the evidence (evidence-bearing documents are in the examined top 3 for 0.910 of the 62
  claims with evidence) but it adds topically related, unannotated abstracts that the judge accepts as
  SUPPORT. The oracle also reads fewer abstracts (114 vs 300).
* **Judge-only has a higher macro-F1 (0.504) than the full system (0.487) despite lower accuracy
  (0.50 vs 0.56).** Without documents the model rejects 9 of 21 CONTRADICT claims (F1 0.529) — many
  are inverted versions of well-known findings — while `full` gets only 4 (F1 0.296), because shown an
  abstract on the topic the judge tends to answer SUPPORT. `full` is better on SUPPORT (recall 0.902
  vs 0.585) and NEI (F1 0.526 vs 0.436). Both gaps are within noise (Δ accuracy +0.06 [−0.06, +0.18],
  Δ macro-F1 −0.02 [−0.15, +0.12]); judge-only answers have no evidence and cannot be checked.
* **Single prompt vs pipeline:** same accuracy (0.56); the pipeline's macro-F1 is higher (0.487 vs
  0.450) because the single prompt almost never says CONTRADICT (F1 0.174). Within noise. Four
  single-prompt answers (claims 350, 437, 793, 852) were downgraded to NEI by a citation-parsing bug —
  the model wrote `"Document <id>"` as the cited id and the parser did not strip the label. The frozen
  result stays 0.560 / 0.450; the model's raw answers would score 0.550 / 0.458 (within noise). The
  parser is fixed for future runs (`Docs/EVALUATION.md` §8.5).
* **Ablations (all within noise):** the evidence agent helps a little (+0.03 accuracy, +0.04 macro-F1
  vs `evidence_off`, mostly on NEI: F1 0.526 vs 0.377); the re-ranker helps slightly (+0.02 / +0.02;
  evidence-bearing documents examined 0.910 vs 0.874); k = 5 ≈ k = 3 (one claim differs); k = 1 has the
  same accuracy with fewer distractors but the weakest CONTRADICT F1 (0.214).
* **CONTRADICT is the bottleneck and a judgement problem:** every document-reading variant finds at
  most 4 of the 21 CONTRADICT claims and the oracle finds 3; in `full`, 14 of the 15 CONTRADICT →
  SUPPORT errors examined the contradicting abstract and judged it SUPPORT (section 8).
* **Operationally solid:** 0 errors in 800 claim runs, coverage 1.000, quote validity 1.000, no
  downgraded judgement in the pipeline variants (the 4 single-prompt downgrades above came from the
  since-fixed parsing bug), conflicts in 3–4 % of claims.
* **Latency is not comparable across variants:** judge_only, single prompt, oracle and evidence_off ran
  with a cold LLM cache; `full` reused 30 % of its model requests from the oracle, `rerank_off` 69 %,
  `full_k5` 75 % and `full_k1` 100 % (its 0.007 s per claim is a cache artefact). Retrieval is also
  cached per claim inside the process, so `full`'s 4.83 s excludes the ≈ 1.2 s hybrid + re-rank search.
  The saved cold benchmark (`artifacts/latency/cold_full_k3.json`, 7.9 s per claim, shown in
  section 7) is the realistic figure.

## 7. Live claim check

One live run of the complete pipeline through `ClaimChecker().check(...)`, the same call the CLI and
Streamlit app make. It needs the local Ollama server with `qwen3:4b`; if Ollama is not running, the
cell prints the readiness problems and the fix instead of failing. Its latency depends on the caches
(a repeated claim is served from the LLM response cache), so the next cell first shows the **saved
cold-latency benchmark** (`python -m claim_checker latency --suite smoke --limit 10 --no-cache`:
10 smoke claims, LLM cache off, fresh process, warm-up measured separately).

The claim is SciFact **training** claim 34 (reference abstract 11705328).

In [ ]:
latency_path = ROOT / "artifacts" / "latency" / "cold_full_k3.json"
if latency_path.exists():
    lat = read_json(latency_path)
    w, cc = lat["warmup_seconds"], lat["cache_check"]
    print(f"Saved {lat['created_at']} | {lat['n_claims']} {lat['split']} claims | k = {lat['configuration']['k']} | "
          f"LLM disk cache {'on' if lat['configuration']['llm_disk_cache'] else 'off'} | "
          f"{cc['llm_cache_hits_total']} LLM / {cc['retrieval_cache_hits']} retrieval cache hits | {lat['errors']} errors")
    print(f"Warm-up (excluded): resources {w['resources']:.2f} s + first claim {w['first_claim']:.2f} s = {w['total']:.2f} s")
    stages = ["retrieve", "retrieve.rerank", "select_evidence", "judge", "total"]
    display(pd.DataFrame([{"stage": s, **{k: lat["stages_seconds"][s][k] for k in ("mean", "median", "p95")}}
                          for s in stages if s in lat["stages_seconds"]]).set_index("stage").round(2))
else:
    print("No saved latency benchmark. Create it with:\n"
          "  C:\\Users\\Bhargav\\anaconda3\\python.exe -m claim_checker latency --suite smoke --limit 10 --no-cache")

In [ ]:
from claim_checker.runtime import check_readiness

DEMO_CLAIM = "A deficiency of folate increases blood levels of homocysteine."
live_result = None
if LIVE_DEMOS:
    readiness = check_readiness(cfg)
    if not readiness.ready:
        print("The checker is not ready, so the live demo is skipped:")
        for p in readiness.problems:
            print(f"  - {p['component']}: {p['message']}  -> {p['hint']}")
    else:
        live_result = checker.check(DEMO_CLAIM)   # checker = ClaimChecker(cfg) created in section 4.2
        r = live_result
        print(f"Status: {r.status} | verdict: {r.verdict} | conflict: {r.conflict}")
        if r.status == "ok":
            print("\nExplanation:\n" + textwrap.fill(r.explanation, 110))
            print("\nEvidence quotations (copied from the stored corpus sentences; numbers shown one-based):")
            for q in r.evidence:
                print(f"  [{q.doc_id} sentence {q.sentence_id + 1}] ({q.doc_verdict}) {q.quote}")
            display(pd.DataFrame([{
                "rank": d.rank, "doc_id": d.doc_id, "title": d.title[:70], "score": round(d.retrieval_score or 0, 3),
                "selected (0-based)": d.selected_sentence_ids,
                "verdict": d.judgement.verdict if d.judgement else None,
                "cited (0-based)": d.judgement.cited_sentence_ids if d.judgement else None,
                "confidence (uncalibrated)": d.judgement.confidence if d.judgement else None,
                "reference doc?": d.doc_id in gold_train["34"].reference_doc_ids,
            } for d in r.documents]))
        else:
            print(f"Error ({r.error.error_type}): {r.error.message}\nHint: {r.error.hint}")
        print("\nTimings (s):", r.timings)
        print("LLM calls:", r.metadata.get("llm_calls"), "| cache hits:", r.metadata.get("llm_cache_hits"),
              "| model digest:", (r.metadata.get("model_digest") or "")[:12])
else:
    print("LIVE_DEMOS is False - skipped.")

## 8. Failure analysis

`report` selects failures automatically (`failure_cases.csv`): the `full` variant first, round-robin
across the three classes with **CONTRADICT given a double share**, adding other variants until at
least ten exist. Each case gets a heuristic *earliest failing stage*:

* **operational** — the run ended with an error (no verdict);
* **retrieval miss** — no evidence-bearing abstract was among the k examined;
* **evidence miss** — evidence-bearing abstracts were examined but none of their gold evidence
  sentences was selected;
* **judgement error** — the right evidence was available but the verdict was wrong (for gold NEI: a
  false decisive verdict or a conflict).

The human-reviewed reading of at least ten cases against the source abstracts is in
`failure_analysis.md`.

In [ ]:
fail_csv = RUN_DIR / "failure_cases.csv"
failures_df = None
if fail_csv.exists():
    failures_df = pd.read_csv(fail_csv)
    source = f"saved {fail_csv.relative_to(ROOT)}"
elif by_variant:
    failures_df = pd.DataFrame(select_failures(by_variant))
    source = "selected in memory from the saved predictions (report not yet run)"
if failures_df is not None and len(failures_df):
    print(f"{len(failures_df)} failure cases ({source}){'' if RUN_COMPLETE else ' - PARTIAL RUN'}")
    display(pd.crosstab(failures_df["failure_stage"], failures_df["gold_label"], margins=True))
    display(failures_df[["variant", "claim_id", "claim", "gold_label", "verdict", "status", "failure_stage"]].head(20))
else:
    print("No failure cases available yet.")

fa_md = RUN_DIR / "failure_analysis.md"
if fa_md.exists():
    display(Markdown(fa_md.read_text(encoding="utf-8")))
else:
    print(f"\n{fa_md.relative_to(ROOT)} has not been written yet (human-reviewed analysis of >= 10 failures).")

**A judgement error seen during development** (training claim 2, not part of the final split): for
*"1 in 5 million in UK have abnormal PrP positivity."* (gold CONTRADICT) the model answered SUPPORT
although its own reason stated that the abstract's prevalence (493 per million) is much higher than
1 in 5 million. The gold evidence sentence is printed below from the corpus. This is the typical
CONTRADICT failure of a small model: a numerical comparison it describes correctly but does not turn
into the right label.

In [ ]:
g2 = gold_train["2"]
for d, ids in g2.evidence_by_doc.items():
    print(f"Claim {g2.claim_id} ({g2.label}): {g2.claim}")
    print(f"Document {d}: {docs[d].title}")
    for i in ids:
        print(f"  gold evidence, sentence {i + 1}: {docs[d].sentences[i]}")

### 8.1 Failure-analysis summary

Human reading of all 44 `full` failures against the source abstracts
(`artifacts/runs/final_v1/failure_analysis.md`, shown in full by the cell above, with 19 detailed cases
including 9 CONTRADICT). Earliest responsible stage (unit: claims):

| Stage | Heuristic | Human reading | Gold S / C / N |
|---|---|---|---|
| operational | 0 | 0 | 0 / 0 / 0 |
| retrieval miss | 1 | 1 | 0 / 1 / 0 |
| evidence miss | 5 | 5 | 2 / 3 / 0 |
| judgement error | 38 | 31 | 1 / 13 / 17 |
| annotation ambiguity | – | 7 | 1 / 0 / 6 |
| **total** | 44 / 100 | 44 / 100 | 4 / 41 · 17 / 21 · 23 / 38 |

Gold → predicted: NEI → SUPPORT 23, CONTRADICT → SUPPORT 15, CONTRADICT → NEI 2, SUPPORT → NEI 2,
SUPPORT → CONTRADICT 2.

Main CONTRADICT failure patterns:

* **Polarity / direction inversion** (10 of the 17 CONTRADICT failures): the model's own reason states
  the opposite of its verdict — claim 516 (CRP): "… Thus, the claim that high CRP reduces risk is
  contradicted." → SUPPORT; claim 180 (TDP-43): "abolishes … neuronal loss" read as support for
  "increased neuronal loss". Across all 44 failures the reason argues for another label in 14.
* **Numeric / comparative:** claim 922 — hazard ratio 0.79 for progression "indicating faster
  progression"; claim 759 — a comparative modelling result summarised as plain support.
* **Evidence misses:** empty selection on an on-topic abstract (142) or a selection without the
  decisive sentence fragment and conclusion (338).
* **Distractor-driven conflict:** claim 1029 — two correct CONTRADICT decisions cancelled by one
  spurious SUPPORT on an unannotated abstract.

On NEI claims the dominant pattern is over-calling SUPPORT from topically related abstracts (18 of
the 23 NEI → SUPPORT errors come only from uncited, unannotated abstracts), via entity or population
substitution (e.g. claim 268: a bile-acid abstract taken as evidence about cold exposure) or
over-inference; 7 failures are annotation ambiguities (e.g. claim 1099 *Statins decrease blood
cholesterol*: uncited abstracts do support it).

One concrete improvement per category (a future experiment `final_v2`, not applied here):
judgement — put the reason (and explicit claim/evidence direction fields) before the verdict and add
a verdict–reason consistency re-ask; entity substitution — require decisive citations to mention the
claim's key entities; evidence — re-ask on an empty selection for the top-ranked abstract and always
include the conclusion sentence; numeric — ratio rules (< 1 = reduced risk) and claim decomposition;
retrieval — a biomedical re-ranker or score fusion.

**The prompts were frozen before `final_v1` and were not changed after these failures were seen**
(prompt hashes in the run manifest still match `prompts/`). Any prompt change must be evaluated as a
new run, not presented as untouched evaluation on this subset.

## 9. Reproducibility record

Everything needed to identify the experiment: source revisions and hashes (§1), frozen split hashes
(§3), the index fingerprint (§4), the model digest, prompt hashes and configuration fingerprint stored
in the run manifest.

In [ ]:
record = {
    "MTEB scifact revision": source_manifest["mteb_corpus"]["revision"],
    "scifact_entailment builder revision": source_manifest["builder_script"].get("revision"),
    "SciFact archive sha256": source_manifest["archive"]["sha256"],
    "final split ids sha256": splits["final"]["ids_sha256"],
    "index fingerprint": index_manifest["fingerprint"],
    "prompt hashes (current files)": checker.prompts.hashes(),
}
if run_manifest:
    ident = run_manifest["identity"]
    record.update({
        f"{RUN_ID} model": f"{ident['model']['name']} @ {ident['model']['digest']}",
        f"{RUN_ID} prompt hashes": ident["prompt_hashes"],
        f"{RUN_ID} config fingerprint": ident["config_fingerprint"],
        f"{RUN_ID} index fingerprint": ident["index_fingerprint"],
        f"{RUN_ID} environment": run_manifest.get("environment"),
        "prompts unchanged since the run": ident["prompt_hashes"] == checker.prompts.hashes(),
    })
display(pd.Series(record, name="value").to_frame())

## 10. Lab citations

The original production implementation was written independently. After locating the actual
Module 2 and 3 notebooks, §12 was added on 23 October 2026 with traceable adaptations:

| Location | Actual source (one-based notebook cell) | Adaptation |
|---|---|---|
| §12a | Module 2, `4_Structured_Generation_Lab.ipynb`, cells 36–37, section 4.1 | Pydantic schema, Literal enum and JSON postvalidation, adapted from review sentiment to evidence-backed verdicts. Production uses Ollama JSON-schema decoding rather than Outlines. |
| §12b | Module 3, `1_RAG_Pipeline_Lab.ipynb`, cell 14 | Regex tokenization and BM25Okapi search over real SciFact abstracts, with deterministic ties. |
| §12b | Same Module 3 notebook, cells 16 and 24 | Two-stream reciprocal rank fusion and Recall@k, generalized to ranked IDs and multiple annotated reference papers. |
| §5 | Module 4 multi-agent lab | Conceptual topic mapping only: its notebook was not located, so no code adaptation is claimed. |

Exact local source paths, differences and the boundary between new adaptations and the
independent production code are recorded in `Docs/LAB_REUSE.md`. The new appendix does
not retrospectively establish lab reuse in the original build.


## 11. Limitations and future work

**Limitations** (full list in `Docs/LIMITATIONS.md`):

* Evidence is limited to **5,183 SciFact abstracts**; NEI means "this collection is insufficient",
  not "false". Abstracts only — no full texts, tables or study-quality weighting.
* **Unannotated abstracts are not proven irrelevant**, so evidence precision is computed on annotated
  abstracts only.
* **Small, 4-bit quantised model** (`qwen3:4b`, Q4_K_M, `think=false`): errors with numbers, negation
  and direction of effect; CONTRADICT is the hardest class. Model-written reasons can contain slips;
  quotations are always exact.
* **Confidence is self-reported and uncalibrated.**
* **Conflict → NEI** is a prototype convention; a decisive verdict without a citation is downgraded.
* **Re-ranker domain mismatch** (MS MARCO web search vs biomedical claims).
* **Context limits**: prompts that would not fit the 8,192-token context are refused
  (`context_overflow`) instead of being silently truncated; the evidence agent windows long abstracts.
* **Environment deviation**: verified on Python 3.14.6 (conda base) rather than the planned 3.11 with
  `uv`; exact versions in `requirements.lock.txt`.
* Final test set of 100 claims (21 CONTRADICT): small differences between variants are not
  meaningful on their own.
* Upstream terms: SciFact claims CC BY 4.0 and abstracts ODC-By 1.0 (allenai/scifact); the
  `allenai/scifact_entailment` card is tagged CC BY-NC 2.0; BGE MIT; MiniLM cross-encoder Apache-2.0;
  Qwen3 Apache-2.0. Verify before redistribution.

**Future work**

1. **LoRA judge (brief's stretch goal):** fine-tune a small Qwen model (e.g. Qwen2.5-0.5B) as the judge
   on the SciFact training split only, through a separate Transformers/PEFT environment, and compare it
   with the same base model untuned on the same frozen 100 claims.
2. **Climate-FEVER out-of-domain test (stretch goal):** build a separate evidence collection from
   `tdiggelm/climate_fever`, document its label mapping (including its DISPUTED label) and evaluate
   separately from SciFact.
3. A biomedical cross-encoder or a larger embedding model; claim decomposition for multi-part claims;
   confidence calibration on a held-out set; numeric-comparison checks for the judge.

## 12. Course lab adaptations (23 October 2026)

These additions adapt the actual local labs, with exact source-cell citations in §10.
They are teaching demonstrations; they do not replace the frozen evaluation pipeline,
alter saved results or measure claim-checking accuracy.

### 12a. Structured verdict schema — Module 2 cells 36–37

The lab's product-review enum becomes a three-way scientific verdict. Additional checks
require citations to belong to the supplied context. The synthetic records below exercise
validation only. Schema validity does not establish factual correctness. Production sends
its schema to Ollama's `format` argument and validates again; this example makes no model call.

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, ValidationError


class CourseVerdict(BaseModel):
    model_config = ConfigDict(extra="forbid")
    verdict: Literal["SUPPORT", "CONTRADICT", "NEI"]
    sentence_ids: list[int]
    reason: str


def validate_course_verdict(raw, available_sentence_ids):
    parsed = CourseVerdict.model_validate_json(raw)
    if not set(parsed.sentence_ids) <= set(available_sentence_ids):
        raise ValueError("Citation outside supplied context")
    if parsed.verdict != "NEI" and not parsed.sentence_ids:
        raise ValueError("Decisive verdict requires a source citation")
    return parsed


schema_demo = validate_course_verdict(
    '{"verdict":"NEI","sentence_ids":[],"reason":"Synthetic schema example."}', [0, 1]
)
print("Allowed verdicts:", CourseVerdict.model_json_schema()["properties"]["verdict"]["enum"])
print("Synthetic record:", schema_demo.model_dump())
for invalid_demo in (
    '{"verdict":"TRUE","sentence_ids":[0],"reason":"Invalid enum."}',
    '{"verdict":"SUPPORT","sentence_ids":[999],"reason":"Invalid source ID."}',
):
    try:
        validate_course_verdict(invalid_demo, [0, 1])
    except (ValueError, ValidationError) as exc:
        print("Rejected:", type(exc).__name__)
    else:
        raise AssertionError("Invalid demonstration unexpectedly accepted")

### 12b. BM25, RRF and Recall@k — Module 3 cells 14, 16 and 24

The lab's tokenizer/BM25 score loop is adapted to the real 5,183 stored abstracts.
Its two RRF loops are adapted to ID lists, with deterministic tie-breaking and duplicate
removal. Recall becomes the fraction of reference papers retrieved per claim, so it
supports multiple references. The BM25 illustration uses one fitting TRAIN claim and is
not a benchmark. The RRF illustration uses two genuine sparse ranking streams (title
and full abstract); it demonstrates the fusion rule, not the production dense hybrid.
The production system's BM25/dense/reranker comparison remains in §4.

In [ ]:
import re
from pathlib import Path
from rank_bm25 import BM25Okapi


def course_tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def course_bm25_index(source_docs, title_only=False):
    ids = sorted(source_docs)
    texts = [source_docs[i].title + (
        "" if title_only else " " + " ".join(source_docs[i].sentences)
    ) for i in ids]
    return ids, BM25Okapi([course_tokens(text) for text in texts])


def course_bm25_search(query, index, k=10):
    ids, bm25 = index
    scores = bm25.get_scores(course_tokens(query))
    positions = sorted(range(len(ids)), key=lambda i: (-float(scores[i]), ids[i]))[:k]
    return [ids[i] for i in positions]


def course_rrf(first_ids, second_ids, k_const=60, top_k=10):
    scores = {}
    for rank, doc_id in enumerate(dict.fromkeys(first_ids), start=1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1 / (k_const + rank)
    for rank, doc_id in enumerate(dict.fromkeys(second_ids), start=1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1 / (k_const + rank)
    return sorted(scores, key=lambda doc_id: (-scores[doc_id], doc_id))[:top_k]


def course_reference_recall(ranked_ids, reference_ids, k=3):
    reference_ids = set(reference_ids)
    return len(set(ranked_ids[:k]) & reference_ids) / len(reference_ids) if reference_ids else None


course_plan = read_json(Path(cfg.root) / "artifacts/improvement/lora_science_protocol_v2.json")
course_claim = next(gold_train[i] for i in course_plan["fit_ids"] if gold_train[i].reference_doc_ids)
course_full_index = course_bm25_index(docs)
course_title_index = course_bm25_index(docs, title_only=True)
course_full_ids = course_bm25_search(course_claim.claim, course_full_index)
course_title_ids = course_bm25_search(course_claim.claim, course_title_index)
course_fused_ids = course_rrf(course_full_ids, course_title_ids)
print("TRAIN fitting claim:", course_claim.claim_id, "| corpus papers:", len(docs))
print("Full-abstract BM25:", course_full_ids[:3])
print("Title BM25:", course_title_ids[:3])
print("Two-sparse-stream RRF:", course_fused_ids[:3])
print("Illustrative reference Recall@3:",
      course_reference_recall(course_fused_ids, course_claim.reference_doc_ids))
assert set(course_fused_ids) <= set(course_full_ids) | set(course_title_ids)
del course_full_index, course_title_index

## 13. Later experiments and evaluation provenance

The original tables above remain historical results. The original 300 validation claims
have since been reused across experiments, so they are not a blind test. The current best
confirmed result on that complete population is **80.67% accuracy / 0.795974 Macro-F1**,
below the requested >95% / >0.95. See `Docs/ACCURACY_IMPROVEMENT_2026-10-22.md`.

The latest all-layer LoRA candidate uses paper/text-grouped TRAIN fitting (484 claims),
stopping (160) and auditing (163), with fixed settings and exact source quotations.
Its protocol, learning curve, audit, acceptance gate and current status are in
`Docs/LORA_SCIENCE_EXPERIMENT.md`. No target is claimed from training loss or unit tests.
The general-science collection is evaluated separately; its 15-case developer smoke
test does not establish population accuracy. Module 4 and separate judging guidelines
referenced by P08 are still unavailable, so full course-lab compliance is not claimed.